# Feature Engineering & Feature Selection

A comprehensive guide to transforming raw data into meaningful features and selecting the most impactful ones for model performance.

## Topics Covered
1. **Feature Engineering**: Encoding, Binning, Polynomial Features, Log/Sqrt Transformations, Date Features
2. **Feature Selection**: Filter Methods, Wrapper Methods, Embedded Methods
3. **Handling Missing Values**: Imputation strategies
4. **Feature Scaling**: StandardScaler, MinMaxScaler, RobustScaler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PolynomialFeatures
)
from sklearn.feature_selection import (
    SelectKBest, f_classif, f_regression,
    mutual_info_regression, RFE,
    VarianceThreshold
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## 2. Feature Engineering Techniques

### 2.1 Binning / Discretization
Convert continuous variables into categorical bins.

In [ ]:
# Bin HouseAge into categories
df['AgeGroup'] = pd.cut(df['HouseAge'], bins=[0, 10, 20, 30, 40, 52],
                        labels=['New', 'Recent', 'Mid', 'Old', 'VeryOld'])

print(df['AgeGroup'].value_counts())
df['AgeGroup'].value_counts().plot(kind='bar', title='House Age Groups')
plt.show()

### 2.2 Log & Sqrt Transformations
Handle skewed distributions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].hist(df['Population'], bins=50, edgecolor='black')
axes[0].set_title(f'Original (skew={df["Population"].skew():.2f})')

# Log Transform
df['Population_log'] = np.log1p(df['Population'])
axes[1].hist(df['Population_log'], bins=50, edgecolor='black', color='orange')
axes[1].set_title(f'Log Transform (skew={df["Population_log"].skew():.2f})')

# Sqrt Transform
df['Population_sqrt'] = np.sqrt(df['Population'])
axes[2].hist(df['Population_sqrt'], bins=50, edgecolor='black', color='green')
axes[2].set_title(f'Sqrt Transform (skew={df["Population_sqrt"].skew():.2f})')

plt.tight_layout()
plt.show()

### 2.3 Polynomial Features
Create interaction terms and polynomial features.

In [ ]:
# Create polynomial features from 2 columns
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
sample = df[['MedInc', 'AveRooms']].head(5)

poly_features = poly.fit_transform(sample)
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out())
print("Original shape:", sample.shape)
print("Polynomial shape:", poly_df.shape)
poly_df

### 2.4 Interaction Features

In [ ]:
# Manual interaction features
df['Rooms_per_Household'] = df['AveRooms'] / df['AveOccup']
df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']
df['Population_per_Household'] = df['Population'] / df['AveOccup']

print("New features created:")
print(df[['Rooms_per_Household', 'Bedrooms_per_Room', 'Population_per_Household']].describe())

## 3. Feature Scaling Comparison

In [ ]:
# Compare different scalers
numeric_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
sample_data = df[numeric_cols].head(1000)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].boxplot(sample_data.values)
axes[0].set_xticklabels(numeric_cols, rotation=45)
axes[0].set_title('Original')

for idx, (name, scaler) in enumerate(scalers.items(), 1):
    scaled = scaler.fit_transform(sample_data)
    axes[idx].boxplot(scaled)
    axes[idx].set_xticklabels(numeric_cols, rotation=45)
    axes[idx].set_title(name)

plt.tight_layout()
plt.show()

## 4. Feature Selection Methods

In [ ]:
# Prepare data for feature selection
feature_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
                'AveOccup', 'Latitude', 'Longitude']
X = df[feature_cols]
y = df['MedHouseVal']

### 4.1 Filter Method — Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[feature_cols + ['MedHouseVal']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Correlation with target
print("\nCorrelation with Target (MedHouseVal):")
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False))

### 4.2 Filter Method — Variance Threshold

In [ ]:
selector = VarianceThreshold(threshold=0.1)
selector.fit(X)
print("Features with variance > 0.1:")
for col, var, selected in zip(feature_cols, selector.variances_, selector.get_support()):
    print(f"  {col:20s} variance={var:10.4f}  {'SELECTED' if selected else 'DROPPED'}")

### 4.3 Filter Method — SelectKBest (Mutual Information)

In [ ]:
selector = SelectKBest(score_func=mutual_info_regression, k=5)
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=feature_cols).sort_values(ascending=False)
scores.plot(kind='barh', title='Mutual Information Scores', color='teal')
plt.xlabel('MI Score')
plt.tight_layout()
plt.show()
print(f"\nSelected features: {list(np.array(feature_cols)[selector.get_support()])}")

### 4.4 Wrapper Method — Recursive Feature Elimination (RFE)

In [ ]:
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rfe = RFE(estimator=model, n_features_to_select=5, step=1)
rfe.fit(X, y)

print("RFE Feature Rankings:")
for col, rank, selected in zip(feature_cols, rfe.ranking_, rfe.support_):
    print(f"  {col:20s} rank={rank}  {'SELECTED' if selected else ''}")

### 4.5 Embedded Method — Lasso Regularization (L1)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = Lasso(alpha=0.01, random_state=42)
lasso.fit(X_scaled, y)

coefs = pd.Series(np.abs(lasso.coef_), index=feature_cols).sort_values(ascending=False)
print("Lasso Coefficients (L1 Regularization):")
print(coefs)

coefs.plot(kind='barh', title='Lasso Feature Importance (|Coefficient|)', color='coral')
plt.xlabel('Absolute Coefficient')
plt.tight_layout()
plt.show()

### 4.6 Embedded Method — Tree-based Feature Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest Feature Importances:")
print(importances)

importances.plot(kind='barh', title='Random Forest Feature Importance', color='steelblue')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Summary Comparison

| Method | Type | Pros | Cons |
|--------|------|------|------|
| Correlation | Filter | Fast, simple | Only linear relationships |
| Variance Threshold | Filter | Simple, removes constants | Ignores target |
| Mutual Information | Filter | Captures non-linear | Computationally heavier |
| RFE | Wrapper | Model-aware, accurate | Slow for large datasets |
| Lasso (L1) | Embedded | Auto feature selection | Assumes linear model |
| Tree Importance | Embedded | Handles non-linear | Biased toward high cardinality |